In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
from brax import envs
from brax.io import html
from IPython.display import HTML, display
from flax import serialization
from networks import GCMLP, GC_PPO_Policy
from trajectory_encoder import TaskState, CircularEncoder

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
seed = 42
# seed = 4444
loop_random_key = jax.random.PRNGKey(seed)
loop_random_key, subkey = jax.random.split(loop_random_key)

env = envs.create(env_name="ant", episode_length=4096, backend="mjx", auto_reset=True)
task_encoder = CircularEncoder(v_max=3.0)

policy_network = GC_PPO_Policy(
    hidden_layer_sizes=(64, 64),
    action_dim=env.action_size,
    initial_std=0.1 * jnp.ones(env.action_size),
    kernel_init=jax.nn.initializers.orthogonal(jnp.sqrt(2)),
    kernel_init_final=jax.nn.initializers.orthogonal(0.01),
    activation=nn.softplus,
    # activation=nn.relu,
    final_activation=jnp.tanh,
    learnable_std=False,
    # learnable_std=True,
)

In [ ]:
fake_obs = jnp.zeros(shape=(env.observation_size,))
fake_zs = jnp.zeros(shape=(task_encoder.z_dim,))
dummy_policy_params = policy_network.init(subkey, obs=fake_obs, z=fake_zs)

In [ ]:
with open("./circle/circle_model.msgpack", "rb") as f:
    model_bytes_data = f.read()
    
model_params = serialization.from_bytes(dummy_policy_params, model_bytes_data)


In [ ]:
def play_step_fn(state, task_state):

    action, _ = policy_network.apply(model_params, state.obs, task_state.z)

    next_state = env.step(state, action)
    truncation = next_state.info['truncation']
    done = next_state.done - truncation

    next_task_state = task_encoder.step(
        task_state, 
        jnp.where(
            done, 
            -task_state.deviation, 
            next_state.pipeline_state.x.pos[0, :2] - state.pipeline_state.x.pos[0, :2],
        ), # displacement
        )

    return next_state, next_task_state, done

In [ ]:


# --- 4. 执行 Rollout 并记录轨迹 ---
def run_rollout(v, omega, key):
    states = []
    infos = []
    key, subkey = jax.random.split(key)
    
    state = env.reset(subkey)

    task = jnp.array([v[0], v[1], omega,])
    z = jnp.array([0.0, 0.0, v[0], v[1], omega,])
    task_state = TaskState(
        deviation=jnp.zeros((2,)),
        task=task,
        t=0.0,
        z=z,
        key=key,
        r=1.0,
    )

    
    for i in range(500):
        states.append(state)

        state, task_state, done = jax.jit(play_step_fn)(state, task_state)
        infos.append(done) # record done info

        # print(int(i + 1), "/ 500")
        
        
    return states, infos

In [ ]:
rng = jax.random.PRNGKey(8848)
# 采样任务并记录轨迹
print("simulating trajectory...")
trajectory_states, trajectory_infos = run_rollout(jnp.array([2.0, 0]), 0.5, rng)

In [ ]:
print("visualizing")
vis_html = html.render(env.sys, [s.pipeline_state for s in trajectory_states])
# display(HTML(vis_html))

In [ ]:
xs = jnp.array([s.pipeline_state.x.pos[0, 0] for s in trajectory_states])
ys = jnp.array([s.pipeline_state.x.pos[0, 1] for s in trajectory_states])

In [ ]:
plt.figure(figsize=(jnp.max(xs)- jnp.min(xs), jnp.max(ys)- jnp.min(ys)))
plt.scatter(
    xs[100:], ys[100:]
    )
plt.show()

In [ ]:
# with open("visualize.html", "w") as f:
#     f.write(vis_html)